# L09 · 向量化核心习惯

**预计时长**：60 min | **难度**：⭐⭐⭐⭐ | **前置**：L08

## 本节目标
1. 理解为什么量化必须向量化（速度差 100 倍）
2. NumPy ndarray 基础 + 广播机制
3. `np.where` 替代 if-else 循环
4. **apply / map / itertuples 的边界**（什么时候 NOT 用）
5. 性能基准测试 `%timeit`

## 第 1 段：为什么量化必须向量化

量化场景下，你要处理的不是"一个数字"，而是：
- 1000 只股票 × 10 年 × 250 交易日 = **250 万个数据点**
- 用 for 循环逐个处理 → 几分钟到几小时
- 用向量化 → 几秒到几十秒

**Python 慢的本质**：每个操作都要经过解释器、类型检查、动态分发。NumPy 用 C 实现，绕过这些开销。

### ⚠️ 监管提示（2025.7.7 起）
中国证监会《程序化交易管理实施细则》已正式实施：每秒申报 ≥ 300 笔或单日笔数较高的，认定为**高频交易**，监管对其收取差异化收费、加强监测。**向量化提速用于研究与回测没问题**，但真要上实盘"刷单"，请先确认符合监管要求。Phase 1 的内容全部限于研究与回测范畴。

### 核心原则
> **永远优先使用 Pandas/NumPy 的"整列运算"，不要 for 循环遍历行。**

L01-L08 你已经在用（`.rolling()`、`.shift()`、`.cumprod()` 等都是向量化），本节让你**意识到**这个习惯，并测出速度差。

In [ ]:
import sys
from pathlib import Path

# 自动定位 phase1_foundation 目录 + project root（兼容两种 jupyter 启动位置）
_cwd = Path.cwd()
_p1 = _cwd if (_cwd / '_data.py').exists() else (_cwd / 'learning' / 'phase1_foundation')
sys.path.insert(0, str(_p1))
_proj = _p1.parent.parent if _p1.name == 'phase1_foundation' else _p1
if (_proj / 'qtrader' / '__init__.py').exists():
    sys.path.insert(0, str(_proj))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import _style
from _data import get_stock_data

## 第 2 段：NumPy ndarray 基础

In [ ]:
# NumPy 的核心数据结构：ndarray（n 维数组）
a = np.array([1, 2, 3, 4, 5], dtype='float64')
b = np.array([10, 20, 30, 40, 50])

# 整数组运算：自动逐元素
print("a + b =", a + b)
print("a * 2 =", a * 2)  # 标量广播
print("a > 3 =", a > 3)  # 布尔数组
print("a[a > 3] =", a[a > 3])  # 布尔索引

# 和 list 对比
py_list = [1, 2, 3, 4, 5]
# py_list + py_list  # 这是拼接 [1,2,3,4,5,1,2,3,4,5]，不是逐元素加！
print("list + list =", py_list + py_list)

## 第 3 段：广播机制（broadcasting）

In [ ]:
# 形状不同的数组相加，NumPy 会"广播"成相同形状
matrix = np.array([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9]])  # shape (3, 3)
row = np.array([10, 20, 30])  # shape (3,)

# row 被广播成 (3, 3)：每行都加 [10,20,30]
print("matrix + row =\n", matrix + row)

col = np.array([[100], [200], [300]])  # shape (3, 1)
print("\nmatrix + col =\n", matrix + col)

In [ ]:
# 实战：1000 只股票 10 年日收益，每个股票累乘
rng = np.random.default_rng(42)
rets = rng.normal(0.0005, 0.02, size=(2520, 1000))  # 2520 日 × 1000 股
print(f"数据 shape: {rets.shape}")

# 每只股票的累计收益（沿 axis=0 累乘）
cum = np.cumprod(1 + rets, axis=0) - 1
print(f"末值 shape: {cum[-1].shape}")
print(f"前 5 只末值: {cum[-1, :5]}")

## 第 4 段：`np.where` 替代 if-else 循环

In [ ]:
# 任务：把 rets 中 > 0 的标 1，<= 0 的标 0
# ❌ 错误做法：双层 for
# ✅ 正确做法：np.where

signals = np.where(rets > 0, 1, 0)
print(f"信号 shape: {signals.shape}, 均值（即胜率）: {signals.mean():.4f}")

# 多条件组合：np.select
conditions = [rets > 0.01, rets < -0.01]
choices = [2, -2]  # 大涨标 2，大跌标 -2
multi_signal = np.select(conditions, choices, default=0)
print("信号分布：", np.unique(multi_signal, return_counts=True))

## 第 5 段：for 循环 vs 向量化 速度对比

用 `%timeit` 实测。这个对比你应该**亲眼看到**，让肌肉记忆刻进去。

In [ ]:
# 准备数据：单只股票 2520 日收益率
byd = get_stock_data('002594').set_index('date')
rets = byd['close'].pct_change().dropna().values  # 转 ndarray
print(f"数据点数: {len(rets)}")

In [ ]:
# 任务：算每个时间点的累计收益率
# 版本 A：for 循环
def cum_with_loop(rets: np.ndarray) -> np.ndarray:
    result = np.empty_like(rets)
    cum = 1.0
    for i, r in enumerate(rets):
        cum *= (1 + r)
        result[i] = cum
    return result

# 版本 B：向量化（np.cumprod）
def cum_vectorized(rets: np.ndarray) -> np.ndarray:
    return np.cumprod(1 + rets, axis=0)

# 验证两者结果一致
a = cum_with_loop(rets)
b = cum_vectorized(rets)
assert np.allclose(a, b), "结果不一致！"
print("✓ 两版本结果一致")

In [ ]:
# 速度对比
import timeit
n_loop = 1000

t_loop = timeit.timeit(lambda: cum_with_loop(rets), number=n_loop)
t_vec  = timeit.timeit(lambda: cum_vectorized(rets), number=n_loop)
print(f"for 循环:    {t_loop*1000:>8.1f} ms / {n_loop} 次 = {t_loop/n_loop*1000:.3f} ms/次")
print(f"向量化:      {t_vec*1000:>8.1f} ms / {n_loop} 次 = {t_vec/n_loop*1000:.3f} ms/次")
print(f"加速比:      {t_loop/t_vec:>8.1f} ×")

In [ ]:
# 用 magic 命令更简洁（仅 notebook 可用）
%timeit cum_with_loop(rets)
%timeit cum_vectorized(rets)

## 第 6 段：apply / map / itertuples 的边界

Pandas 的 `apply` 很方便，但**慢**。原则：

| 场景 | 用什么 | 速度 |
|------|-------|------|
| 单列变换（如 abs） | `.abs()` / `.astype()` 内置方法 | 最快 |
| 多列组合运算 | 整列算术 + np.where | 最快 |
| 复杂自定义函数 | `.apply()` / `.map()` | 中等 |
| 必须逐行依赖前一行 | `for` 循环 / `itertuples()` | 最慢（但有时不可避免） |

**判断标准**：能用整列算术就别用 apply。apply 里再套循环就是灾难。

In [ ]:
df = pd.DataFrame({'a': np.random.randn(10000), 'b': np.random.randn(10000)})

# 任务：算 a^2 + b^2
def f(row):
    return row['a']**2 + row['b']**2

# 三种写法（取消注释逐个 %timeit）
%timeit df.apply(f, axis=1)              # ❌ 最慢
%timeit df['a']**2 + df['b']**2          # ✅ 最快

# 典型输出（数字因机器而异）：
# apply:      ~150 ms
# 整列算术:   ~0.5 ms
# 加速比:     ~300×

## 第 7 段：随堂小练

### 小练：把给定 for 循环改写成向量化

下面这个函数计算"过去 N 日中上涨日数"（用于 RSI 等指标）。把它改写成向量化版本，并用 %timeit 对比。

In [ ]:
# 原版：for 循环
def up_days_loop(rets: np.ndarray, window: int = 10) -> np.ndarray:
    result = np.full(len(rets), np.nan)
    for i in range(window, len(rets)):
        result[i] = (rets[i-window:i] > 0).sum()
    return result

# TODO: 你的向量化版本（约 2 行）
def up_days_vec(rets: np.ndarray, window: int = 10) -> np.ndarray:
    pass



# 验证 + 测速
rng = np.random.default_rng(0)
test_rets = rng.normal(0, 0.02, 1000)
a = up_days_loop(test_rets)
b = up_days_vec(test_rets)
# assert np.allclose(a[10:], b[10:], equal_nan=True)  # 取消注释验证

## 第 8 段：课后练习 + 下节预告

### 📝 `exercises/ex09.py`
1. 把"计算 RSI"的 for 循环版改成纯向量化（用 `np.where` + `rolling`）
2. 写 `batch_cumulative_nav(rets_matrix)`：输入 (N_days, N_stocks) 矩阵，返回每股票累计净值（一条向量化语句搞定）
3. 找出 qtrader 代码里一处可以用向量化优化的地方，写优化前后两版

### 🔮 下节 L10：综合项目（毕业考核）
用前面所有技能（数据/清洗/复权/统计/指标/向量化）产出一份完整的股票分析 HTML 报告。

## 第 9 段：Jupyter tip 🔧
- `%timeit` 自动多次取平均；`%%timeit` 测整 cell
- `%prun my_func()` 行级性能分析（找出最慢的那行）
- `%load_ext line_profiler` + `%lprun -f my_func my_func()` 更细
- NumPy 大数组运算前用 `np.ascontiguousarray` 整理内存布局，可再快 1.5×